In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import LightSource
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams['figure.figsize'] = (12, 10)
plt.rcParams['font.size'] = 12

# Metadynamics on the Müller–Brown Surface

### What this notebook does
- Recaps biasing with collective variables (CVs) and metadynamics basics.
- Shows how to load/reshape the potential like gnuplot (column-major data).
- Visualizes trajectory, hills, and compares reconstructed FES to the original potential.

### When to run (order)
1) From `day7_path_meta_dynamics`: `bash make_potential.sh` (once).
2) Run MD/metadynamics: `cd Exercise/1_metadynamics && ../../plumed/bin/plumed  pesmd < ../input`.
3) Reconstruct FES: `../../plumed/bin/plumed sum_hills --hills HILLS --outfile fes.dat --min -3.0,-1.5 --max 2.0,3.5 --bin 99,99`.
4) Then run this notebook top-to-bottom.

### Quick theory (8.2.1–8.2.4 condensed)
- Biasing methods add a history-dependent potential on CVs to overcome timescale limits.
- Metadynamics: deposit Gaussians (height H, widths W, stride τ); long-time bias → −F(CV).
- Path metadynamics (covered in next notebook): use a path-CV to reduce dimensionality; adaptive nodes relax toward MFEP.
- Model system: Müller–Brown PES with harmonic walls to confine sampling.

### Questions to explore
1) Standard MD: what does it sample? How to change behavior without enhanced sampling?
2) Metadynamics: how do Gaussian width/height/stride affect crossings and FES quality?
3) Fixed path-CV: interpret 1D σ width; effect of tube potential; final path/FES.
4) Adaptive path-CV: vary path-update vs hill stride; tube strength; half-life trends.

## 1. Load Data

In [ ]:
# Load the potential energy surface
# Data stored column-major: for each x, all y are listed (gnuplot style)
potential_data = np.loadtxt('Exercise/potential.dat', comments='#')
x_vals = np.unique(potential_data[:, 0])
y_vals = np.unique(potential_data[:, 1])
X, Y = np.meshgrid(x_vals, y_vals)
# Reshape with y varying fastest, then transpose to match meshgrid
Z_potential = potential_data[:, 2].reshape(len(x_vals), len(y_vals)).T
Z_shifted = Z_potential - Z_potential.min()

# Load metadynamics trajectory
colvar = np.loadtxt('Exercise/1_metadynamics/colvar.out')
time = colvar[:, 0]
cv_x = colvar[:, 1]
cv_y = colvar[:, 2]
bias = colvar[:, 3]

# Load HILLS file
hills = np.loadtxt('Exercise/1_metadynamics/HILLS')
hills_time = hills[:, 0]
hills_x = hills[:, 1]
hills_y = hills[:, 2]
hills_sigma_x = hills[:, 3]
hills_sigma_y = hills[:, 4]
hills_height = hills[:, 5]
hills_bias = hills[:, 6]

# Load free energy surface
fes_data = np.loadtxt('Exercise/1_metadynamics/fes.dat', comments='#')
fes_x = np.unique(fes_data[:, 0])
fes_y = np.unique(fes_data[:, 1])
FES_X, FES_Y = np.meshgrid(fes_x, fes_y)
# FES stored with x varying slowest; reshape to (nx, ny) then transpose
FES = fes_data[:, 2].reshape(len(fes_x), len(fes_y))
FES_shifted = FES - FES.min()

print(f"Trajectory length: {len(time)} frames")
print(f"Number of hills deposited: {len(hills)}")
print(f"Hill height: {hills_height[0]:.3f} K")
print(f"Hill width (σ_x, σ_y): ({hills_sigma_x[0]:.3f}, {hills_sigma_y[0]:.3f})")
print(f"Deposition pace: every {int(hills_time[1] - hills_time[0])} steps")

## 2. Visualize Metadynamics Trajectory

Unlike standard MD, metadynamics explores the entire potential energy surface by filling in energy basins.

In [ ]:
# Plot trajectory on the potential
fig, ax = plt.subplots(figsize=(9, 6))

# Plot potential as background
levels = np.linspace(Z_potential.min(), 0, 30)
contour = ax.contourf(X, Y, Z_potential, levels=levels, cmap='viridis', alpha=0.8)
ax.contour(X, Y, Z_potential, levels=levels, colors='white', alpha=0.3, linewidths=0.5)

# Plot trajectory colored by time
scatter = ax.scatter(cv_x, cv_y, c=time, s=2, cmap='hot', alpha=0.6, zorder=10)

# Plot hills locations
ax.scatter(hills_x, hills_y, c='cyan', s=10, alpha=0.3, 
          marker='x', label='Hill depositions', zorder=15)

# Mark start and end
ax.plot(cv_x[0], cv_y[0], 'go', markersize=15, label='Start', zorder=20)
ax.plot(cv_x[-1], cv_y[-1], 'r*', markersize=20, label='End', zorder=20)

cbar1 = plt.colorbar(contour, ax=ax, label='Potential Energy (K)', pad=0.12)
cbar2 = plt.colorbar(scatter, ax=ax, label='Time (steps)', fraction=0.046, pad=0.04)

ax.set_xlabel('cv.x')
ax.set_ylabel('cv.y')
ax.set_title('Metadynamics Trajectory with Hill Depositions')
ax.set_xlim([-1.5, 1.5])
ax.set_ylim([-0.5, 2.5])
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Evolution of the Bias Potential

The bias potential grows over time as more hills are deposited. Let's visualize how it evolves.

In [ ]:
# Plot bias potential evolution along trajectory
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 4), sharex=True)

# Plot CV evolution
ax1.plot(time, cv_x, label='cv.x', alpha=0.7)
ax1.plot(time, cv_y, label='cv.y', alpha=0.7)
ax1.set_ylabel('Collective Variables')
ax1.set_title('Evolution of CVs and Bias Potential')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot bias evolution
ax2.plot(time, bias, color='red', linewidth=0.8)
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax2.set_xlabel('Time (steps)')
ax2.set_ylabel('Bias Potential (K)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze hill deposition over time
fig, ax = plt.subplots(2, 2, figsize=(10, 8))
ax = ax.flatten()

# Hill locations over time
scatter = ax[0].scatter(hills_x, hills_y, c=hills_time, s=30, 
                     cmap='viridis', alpha=0.6, edgecolors='black', linewidth=0.5)
ax[0].set_xlabel('cv.x')
ax[0].set_ylabel('cv.y')
ax[0].set_title('Spatial Distribution of Hills')
ax[0].set_xlim([-1.5, 1.5])
ax[0].set_ylim([-0.5, 2.5])
plt.colorbar(scatter, ax=ax[0], label='Time (steps)')
ax[0].grid(True, alpha=0.3)

# Plot biasing energy
x0 = hills_x[:, None, None]
y0 = hills_y[:, None, None]
sx = hills_sigma_x[:, None, None]
sy = hills_sigma_y[:, None, None]
h  = hills_height[:, None, None]

# Gaussian formula
bias_energy = np.sum(h * 
                np.exp(-(((X - x0)**2) / (2 * sx**2) + ((Y - y0)**2) / (2 * sy**2)))
                , axis=0)
levels = np.linspace(0, bias_energy.max(), 25)
contour = ax[1].contourf(X, Y, bias_energy, levels=levels, cmap='viridis')
plt.colorbar(contour, ax=ax[1], label='Potential Energy (K)')
ax[1].set_xlabel('cv.x')
ax[1].set_ylabel('cv.y')
ax[1].set_title('Total biasing energy')
ax[1].grid(True, alpha=0.3)
ax[1].set_xlim(-1.5, 1.5)
ax[1].set_ylim(-0.5, 2.5)

levels = np.linspace(0, 250, 25)
contour = ax[2].contourf(X, Y, Z_shifted, levels=levels, cmap='viridis')
plt.colorbar(contour, ax=ax[2], label='Potential Energy (K)')
ax[2].set_xlabel('cv.x')
ax[2].set_ylabel('cv.y')
ax[2].set_title('Original potential')
ax[2].grid(True, alpha=0.3)
ax[2].set_xlim(-1.5, 1.5)
ax[2].set_ylim(-0.5, 2.5)

summed = Z_shifted + bias_energy
levels = np.linspace(0, 250, 25)
contour = ax[3].contourf(X, Y, summed, levels=levels, cmap='viridis')
plt.colorbar(contour, ax=ax[3], label='Potential Energy (K)')
ax[3].set_xlabel('cv.x')
ax[3].set_ylabel('cv.y')
ax[3].set_title('Original potential + bias')
ax[3].grid(True, alpha=0.3)
ax[3].set_xlim(-1.5, 1.5)
ax[3].set_ylim(-0.5, 2.5)

plt.tight_layout()
plt.show()

## 4. Reconstructed Free Energy Surface

One of the key outputs of metadynamics is the reconstructed free energy surface (FES). 

In the long-time limit, the negative of the bias potential converges to the free energy:

$$F(\mathbf{s}) \approx -V_{\text{bias}}(\mathbf{s}, t\to\infty)$$

In [ ]:
# Plot the reconstructed free energy surface
fig = plt.figure(figsize=(12, 4))

# 2D contour plot
ax1 = fig.add_subplot(131)
# Shift FES to have minimum at 0
FES_shifted = FES - FES.min()
levels_fes = np.linspace(0, 150, 25)
contour = ax1.contourf(FES_X, FES_Y, FES_shifted, levels=levels_fes, cmap='viridis')
ax1.contour(FES_X, FES_Y, FES_shifted, levels=levels_fes, colors='white', 
           alpha=0.3, linewidths=0.5)
plt.colorbar(contour, ax=ax1, label='Free Energy (K)')
ax1.set_xlabel('cv.x')
ax1.set_ylabel('cv.y')
ax1.set_title('Reconstructed Free Energy Surface\n(from Metadynamics)')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-1.5, 1.5)
ax1.set_ylim(-0.5, 2.5)

# Original potential for comparison
ax2 = fig.add_subplot(132)
Z_shifted = Z_potential - Z_potential.min()
levels_pot = np.linspace(0, 150, 25)
contour2 = ax2.contourf(X, Y, Z_shifted, levels=levels_pot, cmap='viridis')
ax2.contour(X, Y, Z_shifted, levels=levels_pot, colors='white', 
           alpha=0.3, linewidths=0.5)
plt.colorbar(contour2, ax=ax2, label='Potential Energy (K)')
ax2.set_xlabel('cv.x')
ax2.set_ylabel('cv.y')
ax2.set_title('Original Potential Energy Surface\n(Ground Truth)')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(-1.5, 1.5)
ax2.set_ylim(-0.5, 2.5)

plt.tight_layout()
plt.show()

#### Comparison: Reconstructed FES vs Original Potential

Now we directly compare the reconstructed free energy surface from metadynamics with the original potential energy surface. The FES should closely match the potential if:
- The simulation was run long enough
- The bias was added slowly enough
- The collective variables adequately describe the system

**Left panel**: Overlay of FES (filled contours) and original potential (red contour lines)  
**Right panel**: Difference map showing where the reconstruction deviates from the true potential

In [ ]:
# Direct comparison: FES vs Original Potential overlaid
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))

# Left: Overlay contours
Z_shifted = Z_potential - Z_potential.min()
FES_shifted = FES - FES.min()

# Plot FES filled contours
contour_fes = ax1.contourf(FES_X, FES_Y, FES_shifted, levels=20, cmap='viridis', alpha=0.6)
# Overlay original potential as lines
contour_pot = ax1.contour(X, Y, Z_shifted, levels=20, colors='red', linewidths=1.5, alpha=0.8)
ax1.clabel(contour_pot, inline=True, fontsize=8, fmt='%1.0f')
plt.colorbar(contour_fes, ax=ax1, label='Free Energy (K)')
ax1.set_xlabel('cv.x')
ax1.set_ylabel('cv.y')
ax1.set_title('FES (filled) vs\n Original Potential (red lines)')
ax1.set_xlim([-1.5, 1.5])
ax1.set_ylim([-0.5, 2.5])
ax1.grid(True, alpha=0.3)

# Right: Difference map
# Interpolate FES to match potential grid if needed
from scipy.interpolate import RegularGridInterpolator
interp = RegularGridInterpolator((fes_y, fes_x), FES, bounds_error=False, fill_value=np.nan)
points = np.array([Y.ravel(), X.ravel()]).T
FES_interp = interp(points).reshape(X.shape)

diff = (Z_potential - Z_potential.min()) - (FES_interp - FES_interp.min())
levels = np.linspace(diff.min(), 150, 25)
contour_diff = ax2.contourf(X, Y, diff, levels=levels, cmap='viridis')
ax2.contour(X, Y, Z_shifted, levels=15, colors='white', linewidths=0.5, alpha=0.3)
plt.colorbar(contour_diff, ax=ax2, label='FES - Potential (K)')
ax2.set_xlabel('cv.x')
ax2.set_ylabel('cv.y')
ax2.set_title('Difference: \nOriginal Potential - Reconstructed FES')
ax2.set_xlim([-1.5, 1.5])
ax2.set_ylim([-0.5, 2.5])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFES Reconstruction Quality:")
print(f"  Mean absolute error: {np.nanmean(np.abs(diff)):.2f} K")
print(f"  RMS error: {np.sqrt(np.nanmean(diff**2)):.2f} K")

## 5. Sampling Quality Analysis

In [ ]:
# Compare sampling density between MD and metadynamics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))

# Load standard MD for comparison
md_colvar = np.loadtxt('Exercise/0_md/colvar.out')
md_x = md_colvar[:, 1]
md_y = md_colvar[:, 2]

# Standard MD sampling
H_md, xedges, yedges = np.histogram2d(md_x, md_y, bins=50, 
                                       range=[[-1.5, 1.5], [-0.5, 2.5]])
extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]
ax1.contour(X, Y, Z_potential, levels=20, colors='white', alpha=0.5, linewidths=1)
im1 = ax1.imshow(H_md.T, origin='lower', extent=extent, cmap='hot', 
                interpolation='gaussian', alpha=0.8, aspect='auto')
plt.colorbar(im1, ax=ax1, label='Sampling Density')
ax1.set_xlabel('cv.x')
ax1.set_ylabel('cv.y')
ax1.set_title('Standard MD Sampling')
ax1.set_xlim([-1.5, 1.5])
ax1.set_ylim([-0.5, 2.5])

# Metadynamics sampling
H_metad, _, _ = np.histogram2d(cv_x, cv_y, bins=50, 
                               range=[[-1.5, 1.5], [-0.5, 2.5]])
ax2.contour(X, Y, Z_potential, levels=20, colors='white', alpha=0.5, linewidths=1)
im2 = ax2.imshow(H_metad.T, origin='lower', extent=extent, cmap='hot', 
                interpolation='gaussian', alpha=0.8, aspect='auto')
plt.colorbar(im2, ax=ax2, label='Sampling Density')
ax2.set_xlabel('cv.x')
ax2.set_ylabel('cv.y')
ax2.set_title('Metadynamics Sampling')
ax2.set_xlim([-1.5, 1.5])
ax2.set_ylim([-0.5, 2.5])

plt.tight_layout()
plt.show()

# Calculate coverage statistics
threshold = 5  # minimum visits to consider "sampled"
md_coverage = np.sum(H_md > threshold) / (50 * 50) * 100
metad_coverage = np.sum(H_metad > threshold) / (50 * 50) * 100

print(f"\nSampling Coverage (>{threshold} visits per bin):")
print(f"  Standard MD: {md_coverage:.1f}%")
print(f"  Metadynamics: {metad_coverage:.1f}%")
print(f"  Improvement: {metad_coverage/md_coverage:.1f}x better coverage")

## 6. Analysis of Free Energy Minima

In [ ]:
# Find and analyze free energy minima
from scipy.ndimage import minimum_filter

# Find local minima in FES
local_min = minimum_filter(FES_shifted, size=5) == FES_shifted
min_mask = (FES_shifted < 20) & local_min  # Only consider significant minima

min_indices = np.argwhere(min_mask)
min_energies = FES_shifted[min_mask]
min_coords = [(FES_X[idx[0], idx[1]], FES_Y[idx[0], idx[1]]) 
             for idx in min_indices]

# Sort by energy
sorted_idx = np.argsort(min_energies)

print("Free Energy Minima:")
print("------------------")
for i, idx in enumerate(sorted_idx[:5]):  # Show top 5
    x, y = min_coords[idx]
    e = min_energies[idx]
    print(f"Minimum {i+1}: ({x:6.2f}, {y:6.2f})  Energy: {e:6.2f} K")

# Plot minima on FES
fig, ax = plt.subplots(figsize=(6, 5))
contour = ax.contourf(FES_X, FES_Y, FES_shifted, levels=25, cmap='viridis')
ax.contour(FES_X, FES_Y, FES_shifted, levels=25, colors='white', 
          alpha=0.3, linewidths=0.5)

# Mark minima
for i, idx in enumerate(sorted_idx[:5]):
    x, y = min_coords[idx]
    ax.plot(x, y, 'r*', markersize=20, markeredgecolor='white', markeredgewidth=1.5)
    ax.annotate(f'{i+1}', (x, y), xytext=(10, 10), textcoords='offset points',
               fontsize=14, fontweight='bold', color='white',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='red', alpha=0.7))

plt.colorbar(contour, ax=ax, label='Free Energy (K)')
ax.set_xlabel('cv.x')
ax.set_ylabel('cv.y')
ax.set_title('Free Energy Surface with Identified Minima')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusions

From this metadynamics simulation, we learned:

1. **Enhanced Sampling**: Metadynamics successfully explores the entire potential energy surface, including high-energy regions that standard MD cannot access.

2. **Free Energy Reconstruction**: The bias potential converges to provide an accurate estimate of the free energy landscape.

3. **Barrier Crossing**: The method efficiently promotes transitions between energy minima by filling basins with bias potential.

4. **Practical Considerations**:
   - **Hill height** and **width** must be chosen carefully
   - Too large hills → poor resolution
   - Too small hills → slow convergence
   - **Deposition rate** affects exploration vs. accuracy trade-off

### Limitations:

While powerful, standard metadynamics has some drawbacks:
- Choosing appropriate collective variables is crucial and non-trivial
- Convergence can be slow for high-dimensional CV spaces
- The method explores the entire CV space, which may be inefficient for studying specific transitions

In the next notebooks, we'll explore **path metadynamics**, which focuses sampling along specific transition pathways, providing a more efficient approach for studying rare events.